# Kinetic-energy survival from the FNO rollout

Same energy-based stability metric as `kinetic_energy_survival.ipynb` / [arXiv:2506.04528](https://arxiv.org/abs/2506.04528), applied to the L=1 FNO `rollout_mse.npz` from `eval_fno_jacobian_manual_mse.py`.

Does **not** load model weights. Leaves the HINE notebook and script untouched.

- Kinetic energy: $E = \frac{1}{2HW}\sum_{x,y}(u^2+v^2)$.
- Stable while $|E-\mu_E|\le 5\sigma_E$; once failed, stays failed.
- FNO lead count is `horizon - 1` (19 when `horizon=20`).
- Reference $(\mu_E,\sigma_E)$ is the shared cache under `paper_1gpu/jacobian_manual_mse/`.
- Saved fields are normalized by `norm=10`; multiply by that factor before KE.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

_here = Path.cwd().resolve()
_repo = next(p for p in [_here, *_here.parents] if (p / "model.py").is_file())
for _p in (_repo, _repo / "eval"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import matplotlib.pyplot as plt
import numpy as np
import torch

from eval_fno_kinetic_energy_survival import (
    OUT_NPZ,
    REF_CACHE,
    ROLLOUT_NPZ,
    SAVE_DIR,
    SIGMA_MULT,
    energies_from_high_field,
    load_or_build_reference,
    make_wavenumber_tensors,
    survival_curve,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")
print(f"ROLLOUT_NPZ={ROLLOUT_NPZ}")
print(f"REF_CACHE={REF_CACHE}")
print(f"SAVE_DIR={SAVE_DIR}")
assert ROLLOUT_NPZ.is_file(), f"missing {ROLLOUT_NPZ}"
assert REF_CACHE.is_file() or True

In [ ]:
meta = np.load(ROLLOUT_NPZ, allow_pickle=True)
lead = np.asarray(meta["lead"], dtype=np.int64)
norm = float(np.asarray(meta["norm"]))
ar_pred_high = np.asarray(meta["ar_pred_high"])
gt_high = np.asarray(meta["gt_high"])
n_trials, n_leads = ar_pred_high.shape[0], ar_pred_high.shape[1]
print(f"trials={n_trials} leads={n_leads} norm={norm} model_type={meta.get('model_type', 'n/a')}")
meta.close()

Kx_t, Ky_t, invKsq_t, _, _, _ = make_wavenumber_tensors(256, 256, device)
ref_energies, mu_E, sigma_E = load_or_build_reference(Kx_t, Ky_t, invKsq_t, device)
lo = mu_E - SIGMA_MULT * sigma_E
hi = mu_E + SIGMA_MULT * sigma_E
print(f"mu_E={mu_E:.6g} sigma_E={sigma_E:.6g} band=[{lo:.6g}, {hi:.6g}]")

ar_E = energies_from_high_field(ar_pred_high, norm, Kx_t, Ky_t, invKsq_t, device)
gt_E = energies_from_high_field(gt_high, norm, Kx_t, Ky_t, invKsq_t, device)
S_ar = survival_curve(ar_E, mu_E, sigma_E)
S_gt = survival_curve(gt_E, mu_E, sigma_E)
print(f"AR S(1)={S_ar[0]:.4%} S(final)={S_ar[-1]:.4%}")
print(f"GT S(1)={S_gt[0]:.4%} S(final)={S_gt[-1]:.4%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for i in range(n_trials):
    ax.plot(lead, ar_E[i], color="C0", alpha=0.15, lw=0.8)
    ax.plot(lead, gt_E[i], color="C1", alpha=0.10, lw=0.7)
ax.plot(lead, ar_E.mean(axis=0), color="C0", lw=2.5, label="AR mean")
ax.plot(lead, gt_E.mean(axis=0), color="C1", lw=2.5, label="GT mean")
ax.axhline(mu_E, color="k", ls="-", lw=1.2, label=r"$\mu_E$")
ax.axhline(lo, color="k", ls="--", lw=1.0, label=fr"$\mu_E \pm {SIGMA_MULT:g}\sigma_E$")
ax.axhline(hi, color="k", ls="--", lw=1.0)
ax.set_xlabel("lead")
ax.set_ylabel(r"kinetic energy $E$")
ax.set_title("FNO kinetic energy vs lead")
ax.legend(loc="best")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(lead, S_ar, color="C0", lw=2.5, marker="o", markersize=4, label="AR")
ax.plot(lead, S_gt, color="C1", lw=2.0, ls="--", marker="s", markersize=3, label="GT")
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel("lead")
ax.set_ylabel(r"stability rate $S(n)$")
ax.set_title("FNO stability rate")
ax.legend(loc="best")
ax.grid(True, alpha=0.3)
plt.show()
print(f"script outputs also under {SAVE_DIR} / {OUT_NPZ.name}")